# `footprint_attributes.run()` — the single entry point

The other example notebooks (`direction`, `shape`, `position`,
`building_sizes`) each exercise one module's functions directly. In
practice you usually just want "these columns, on this GeoDataFrame",
without caring which module computes what -- that's what
`footprint_attributes.run()` is for: it wires `direction`/`shape`/`position`
together behind a single call, driven by a `config` dict.

```python
result = footprint_attributes.run(
    footprints_gdf,
    config={
        "columns": [...],       # required: exact column names, norm names
                                 # ("EC8", "ASCE7", ...), "position", "bearing",
                                 # or "slenderness"
        "position": {...},      # optional: kwargs forwarded to position()
        "shape": {...},         # optional: kwargs forwarded to shape()
        "direction": {...},     # optional: kwargs forwarded to direction.inertia()
    },
)
```

This notebook runs it on the three real pilot-area datasets (San José,
Guatemala City, Santo Domingo), requesting a representative mix of columns
-- a couple of full norms, the position pipeline, and two code-independent
indices -- and shows the result on one merged map.

## Setup

In [1]:
import os

import geopandas as gpd
import pandas as pd

import FancyFolium

import footprint_attributes

In [2]:
output_path = "data/output"

os.makedirs(output_path, exist_ok=True)

In [3]:
CITY_FILES = {
    "san_jose": "data/san_jose_pilot_region.gpkg",
    "guatemala": "data/guatemala_pilot_region.gpkg",
    "santo_domingo": "data/santo_domingo_pilot_region.gpkg",
}

def load_city(path):
    """Load a pilot-region file into a clean, single-Polygon-per-row GeoDataFrame.

    Always explodes multi-part geometries, then drops (with a warning) any
    row that is null/empty or fails a validity check, instead of silently
    repairing it -- footprint_attributes functions reject invalid/multi-part
    geometries outright, so bad rows must be removed here, not patched.
    """
    gdf = gpd.read_file(path)

    null_mask = gdf.geometry.isna() | gdf.geometry.is_empty
    if null_mask.any():
        print(f"WARNING: {path}: dropping {null_mask.sum()} null/empty geometries")
    gdf = gdf[~null_mask].reset_index(drop=True)

    gdf = gdf.explode(index_parts=False).reset_index(drop=True)

    non_polygon = gdf.geometry.type != "Polygon"
    if non_polygon.any():
        print(f"WARNING: {path}: dropping {non_polygon.sum()} non-Polygon geometries")
    gdf = gdf[~non_polygon].reset_index(drop=True)

    invalid = ~gdf.geometry.is_valid
    if invalid.any():
        print(f"WARNING: {path}: dropping {invalid.sum()} invalid geometries")
    gdf = gdf[~invalid].reset_index(drop=True)

    # position() (part of the config below) requires a real height for
    # every building; fill with this city's own median height.
    gdf["height"] = gdf["height"].fillna(gdf["height"].median())
    return gdf

## A single `run()` call

One config dict requests: the full `EC8` and `GNDTII` norms (each expands
to that code's own parameter columns, plus a `compliance_*` 0-100 score per
parameter), the `position` pipeline (5 columns, computed atomically), the
`bearing` (from `direction.inertia`), and two code-independent compactness
indices. `position`'s `buffer`/`height_column` are forwarded via the
`"position"` sub-dict.

In [10]:
RUN_CONFIG = {
    "columns": [
        "EC8", "GNDTII", "position", "bearing",
        "polsby_popper", "convex_hull_irregularity", 
        "NTC23_setbackRatio"
    ],
    "position": {"buffer": 0.1, "height_column": "height"},
}

san_jose_demo = footprint_attributes.run(load_city(CITY_FILES["san_jose"]), config=RUN_CONFIG)
new_columns = [c for c in san_jose_demo.columns if c != "geometry"]
print(f"{len(new_columns)} columns computed:")
print(new_columns)
san_jose_demo[["relativePosition", "bearing", "EC8_compactness", "GNDTII_beta2_setbackRatio", "NTC23_setbackRatio"]].head()

33 columns computed:
['id', 'city', 'height', 'n_storeys', 'year', 'code_quality', 'structural_system', 'simplified_structural_system', 'roof', 'compliance_EC8_radiusRatio', 'GNDTII_beta2_setbackRatio', 'compliance_GNDTII_beta4_eccentricityRatio', 'compliance_EC8_compactness', 'compliance_GNDTII_beta6_setbackSlenderness', 'GNDTII_beta6_setbackSlenderness', 'polsby_popper', 'EC8_radiusRatio', 'compliance_EC8_eccentricityRatio', 'compliance_GNDTII_beta1_mainShapeSlenderness', 'EC8_compactness', 'compliance_GNDTII_beta2_setbackRatio', 'NTC23_setbackRatio', 'GNDTII_beta4_eccentricityRatio', 'GNDTII_beta1_mainShapeSlenderness', 'EC8_eccentricityRatio', 'convex_hull_irregularity', 'contact_force', 'contact_confinementRatio', 'contact_angularAcc', 'contact_angle', 'contact_height', 'relativePosition', 'bearing']


,relativePosition,bearing,EC8_compactness,GNDTII_beta2_setbackRatio,NTC23_setbackRatio
0,confined,14.395737,1.000000,0.000000,0.000000
1,corner,3.377607,0.999820,0.000000,0.000000
2,confined,24.013840,0.999880,0.000000,0.000000
3,confined,-85.975534,0.997747,0.027276,0.027276
4,isolated,-79.978056,0.835991,0.337829,0.337829


## Run on all three pilot areas

In [5]:
results = {}
for city, path in CITY_FILES.items():
    gdf = load_city(path)
    results[city] = footprint_attributes.run(gdf, config=RUN_CONFIG)
    print(f"{city:15s} {len(results[city]):5d} footprints, {len(results[city].columns)} columns")

san_jose          911 footprints, 33 columns
guatemala         435 footprints, 33 columns
santo_domingo    1463 footprints, 33 columns


## Result

One GeoDataFrame per city, each carrying every column requested above,
saved to `data/output/` and shown on a single merged map (one tab per
city, same layer set in each):

- `relativePosition` -- categorical position class.
- `bearing` -- building orientation (deg from North, range [-90, 90]).
- `EC8_compactness` -- green near 1.0 (convex), red below the 0.95 limit.
- `GNDTII_beta2_setbackRatio` -- green near 0, red above the code limit.
- `polsby_popper` -- green near 1 (circular), red near 0 (elongated).

In [6]:
san_jose_gdf = results["san_jose"]
guatemala_gdf = results["guatemala"]
santo_domingo_gdf = results["santo_domingo"]

san_jose_gdf.to_file(output_path + "/san_jose_run.gpkg")
guatemala_gdf.to_file(output_path + "/guatemala_run.gpkg")
santo_domingo_gdf.to_file(output_path + "/santo_domingo_run.gpkg")

In [7]:
map_columns = [
    "id", "relativePosition", "bearing", "EC8_compactness",
    "GNDTII_beta2_setbackRatio", "polsby_popper",
]
columns_and_layer_names = [
    {
        "column": "relativePosition",
        "layer_name": "Relative Block Position",
        "categorical": True,
        "active": True,
    },
    {
        "column": "bearing",
        "layer_name": "Bearing (deg from North)",
        "cmap": "viridis",
        "vmin": -90,
        "vmax": 90,
        "active": False,
    },
    {
        "column": "EC8_compactness",
        "layer_name": "EC8 Compactness",
        "cmap": "RdYlGn",
        "vmin": 0.5,
        "vmax": 1.0,
        "active": False,
    },
    {
        "column": "GNDTII_beta2_setbackRatio",
        "layer_name": "GNDTII beta2 (setback ratio)",
        "cmap": "RdYlGn_r",
        "vmin": 0.0,
        "vmax": 0.4,
        "active": False,
    },
    {
        "column": "polsby_popper",
        "layer_name": "Polsby-Popper Compactness",
        "cmap": "RdYlGn",
        "vmin": 0.0,
        "vmax": 1.0,
        "active": False,
    },
]

In [8]:
maps = []
for gdf in [san_jose_gdf, guatemala_gdf, santo_domingo_gdf]:
    m = None
    m = FancyFolium.background_layer("google hybrid", m=m)
    m = FancyFolium.background_layer(
        "cartodb light",
        m=m,
        overlay=False,
        active=False,
    )

    for layer in columns_and_layer_names:
        kwargs = {
            "gdf": gdf,
            "layer_name": layer["layer_name"],
            "column": layer["column"],
            "overlay": False,
            "popup": map_columns,
            "active": layer.get("active", False),
            "style": {
                "stroke_color": "black",
                "weight": 1,
                "fillOpacity": 0.5,
            },
            "m": m,
        }

        if layer.get("categorical", False):
            kwargs["categorical"] = True
        else:
            kwargs["cmap"] = layer.get("cmap", "Reds")
            if "vmin" in layer:
                kwargs["vmin"] = layer["vmin"]
            if "vmax" in layer:
                kwargs["vmax"] = layer["vmax"]

        m = FancyFolium.vector_layer(**kwargs)

    maps.append(m)

m = FancyFolium.merge_maps(maps, ["San Jose", "Guatemala", "Santo Domingo"])
m.save(output_path + "/map_run.html")
m